## Initial Set up

In [2]:
import os
import re
import pathlib
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import display

import tifffile
import scipy.io as sio

from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation
from skimage.filters import gaussian, sobel
from skimage.restoration import estimate_sigma
from skimage.draw import disk as draw_disk

import sys

REPO_ROOT = pathlib.Path(r"C:\Users\tkpp2\Documents\FISH-ML2").resolve()
os.chdir(REPO_ROOT)

import fishCore
f = fishCore.Fish(REPO_ROOT / "config.ini")
f.set_model_version("3.50")



IMG_SIZE = 2048
BASE_DIR = Path(r"Z:\Guohao\For Computation group\EXP251009COL")
MAT_PATH = BASE_DIR / "matlabEXP251009COL_251022_1739_all_track.mat"
FINALIZED_DIR = BASE_DIR.parent / f"{BASE_DIR.name}_finalized"

USE_FINALIZED_IGNORE = True
IGNORE_OVERLAP_THRESH = 0.60
MATCH_IOU_THRESH = 0.50

mat = sio.loadmat(MAT_PATH, struct_as_record=False, squeeze_me=False)
track_keys = sorted([k for k in mat.keys() if k.startswith("Track")])

print("Track keys:")
print(track_keys)
print("Finalized dir:", FINALIZED_DIR)


final text_encoder_type: bert-base-uncased


[2026-04-13 17:38:26,289][INFO] CHECK: Model version 3.50 loaded


Track keys:
['Track251009COLA01', 'Track251009COLA02', 'Track251009COLA03', 'Track251009COLB01', 'Track251009COLB02', 'Track251009COLB03', 'Track251009COLC01', 'Track251009COLC02', 'Track251009COLD01', 'Track251009COLD02', 'Track251009COLE01', 'Track251009COLE02', 'Track251009COLF01', 'Track251009COLF02']
Finalized dir: Z:\Guohao\For Computation group\EXP251009COL_finalized


## Helpers

### Image preprocessing

In [3]:
def remove_outliers(img, k=20.0, use_median=False):
    x = img.astype(np.float32)

    if use_median:
        med = np.median(x)
        mad = np.median(np.abs(x - med)) + 1e-6
        sigma = 1.4826 * mad
        thresh = med + k * sigma
    else:
        mean = np.mean(x)
        std = np.std(x)
        thresh = mean + k * std

    x_clipped = np.minimum(x, thresh)
    x_norm = (x_clipped - x_clipped.min()) / (x_clipped.max() - x_clipped.min() + 1e-6)
    return x_norm

def normalize_to_uint8(img):
    return cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def calculate_sigma(img, verbose=False):
    sig = float(estimate_sigma(img, channel_axis=None, average_sigmas=True))
    if verbose:
        print(f"Estimated Gaussian noise standard deviation = {sig:.4f}")
    return sig

def clahe_filter(img, clip_limit=4.0, tile_size=(8, 8)):
    c = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    return c.apply(img)

def laplacian(img, cyt_bilat):
    lap = cv2.Laplacian(img, cv2.CV_64F)
    lap = cv2.convertScaleAbs(lap)
    cyt_blended = cv2.addWeighted(cyt_bilat, 0.8, lap, 0.2, 0)
    return cyt_blended

def edge_preserve_gray(img_uint8, sigma_s, sigma_r=0.4):
    rgb = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)
    out = cv2.edgePreservingFilter(
        rgb,
        flags=1,
        sigma_s=float(max(sigma_s, 1.0)),
        sigma_r=float(sigma_r),
    )
    return cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)

def postproc_mask(m):
    m = morphology.remove_small_objects(m.astype(bool), min_size=200)
    m = morphology.binary_closing(m, footprint=morphology.disk(3))
    m = morphology.binary_opening(m, footprint=morphology.disk(2))
    return m.astype(np.uint8)

def mask_to_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return (xs.min(), ys.min(), xs.max(), ys.max())

def watershed_segment_with_centers(cyt_img, centers):
    thresh = filters.threshold_otsu(cyt_img)
    binary = cyt_img > thresh
    binary = morphology.remove_small_holes(binary, area_threshold=1000)

    dist = ndi.distance_transform_edt(binary)
    markers = np.zeros(cyt_img.shape, dtype=np.int32)

    for i, (cx, cy) in enumerate(centers, start=1):
        xi, yi = int(round(cx)), int(round(cy))
        if 0 <= yi < cyt_img.shape[0] and 0 <= xi < cyt_img.shape[1]:
            markers[yi, xi] = i

    edges = filters.sobel(cyt_img)
    elevation = -dist + 5 * edges
    labels = segmentation.watershed(elevation, markers=markers, mask=binary)

    masks = []
    for i in range(1, np.max(labels) + 1):
        mask = labels == i
        if np.sum(mask) > 0:
            masks.append(postproc_mask(mask))

    return masks


### Filename parsing & GT reconstruction

In [4]:
channel_token_pattern = re.compile(r"w\d+-([^_]+)", re.IGNORECASE)
frame_pattern = re.compile(r"_s(\d+)", re.IGNORECASE)

def parse_channel(filename):
    m = channel_token_pattern.search(Path(filename).name)
    if m is None:
        return None

    token = m.group(1)
    token = re.sub(r"[^A-Za-z0-9]+", "", token).upper()

    if "DAPI" in token:
        return "DAPI"

    for ch in ["647", "594", "555", "488"]:
        if ch in token:
            return ch

    return None

def parse_frame_id(filename):
    m = frame_pattern.search(Path(filename).name)
    return m.group(1) if m else None


def extract_filename(fname):
    if isinstance(fname, str):
        return fname
    if isinstance(fname, bytes):
        return fname.decode()
    if isinstance(fname, np.ndarray):
        if fname.size == 1:
            return extract_filename(fname.item())
        if fname.dtype.kind in ("U", "S"):
            return "".join(np.asarray(fname).ravel().tolist())
        return str(fname.squeeze())
    return str(fname)

def track_key_to_sample_id(track_key):
    m = re.search(r"Track(\d+COL[A-Z]\d{2})", track_key)
    if m is None:
        raise ValueError(f"Could not parse track key: {track_key}")
    return f"EXP{m.group(1)}"

def build_frame_file_map(sample_dir):
    sample_dir = Path(sample_dir)

    tif_files = sorted([
        *sample_dir.glob("*.tif"),
        *sample_dir.glob("*.tiff"),
        *sample_dir.glob("*.TIF"),
        *sample_dir.glob("*.TIFF"),
    ])

    out = {}
    for p in tif_files:
        sid = parse_frame_id(p.name)
        ch = parse_channel(p.name)
        if sid is None or ch is None:
            continue
        out.setdefault(sid, {})[ch] = p

    return out

def open_tiff_file(image_path):
    img = tifffile.imread(str(image_path))
    if img.ndim == 3:
        if img.shape[0] == 1:
            img = img[0]
        elif img.shape[-1] == 1:
            img = img[..., 0]
    return np.asarray(img)

def extract_cells_from_frame(frame_obj):
    cells_container = getattr(frame_obj, "cells", None)

    if not isinstance(cells_container, np.ndarray):
        return []
    if cells_container.size == 0:
        return []

    out = []
    for obj in cells_container.ravel():
        if isinstance(obj, np.ndarray) and obj.shape == (1, 1):
            obj = obj[0, 0]

        mask = getattr(obj, "mask", None)
        pos = getattr(obj, "pos", None)

        if isinstance(mask, np.ndarray) and mask.ndim == 2 and pos is not None:
            out.append((mask, np.asarray(pos)))

    return out

def reconstruct_instance_mask(cells, H, W):
    inst_mask = np.zeros((H, W), dtype=np.int32)

    for label, (mask, pos) in enumerate(cells, start=1):
        mask = np.asarray(mask) > 0
        pos = np.asarray(pos).astype(int).ravel()
        if pos.size < 2:
            continue

        y0, x0 = int(pos[0]), int(pos[1])

        # MATLAB coords are effectively 1-based in this dataset
        y0 -= 1
        x0 -= 1

        h, w = mask.shape
        y0 = max(0, min(H - h, y0))
        x0 = max(0, min(W - w, x0))
        y1 = min(y0 + h, H)
        x1 = min(x0 + w, W)

        inst_mask[y0:y1, x0:x1][mask[:(y1 - y0), :(x1 - x0)]] = label

    return inst_mask

def get_instance_cmap():
    cmap = plt.get_cmap("tab20").copy()
    cmap.set_under("black")
    return cmap


### Choosing cytoplasm channel

In [5]:
def robust_norm(x, p1=1, p99=99):
    x = np.asarray(x, dtype=np.float32)
    lo = np.percentile(x, p1)
    hi = np.percentile(x, p99)
    x = np.clip(x, lo, hi)
    return (x - lo) / max(hi - lo, 1e-6)

def nucleus_boxes_to_centers(bbox_nucleus):
    return [((x1 + x2) / 2.0, (y1 + y2) / 2.0) for x1, y1, x2, y2 in bbox_nucleus]

def score_cyto_channel(
    cyto_img,
    centers,
    smooth_sigma=1.0,
    bg_disk_radius=25,
    annulus_inner_r=8,
    annulus_outer_r=26,
):
    cyto_norm = robust_norm(cyto_img)
    cyto_s = gaussian(cyto_norm, sigma=float(smooth_sigma), preserve_range=True)
    bg = morphology.opening(cyto_s, morphology.disk(int(bg_disk_radius)))
    fg_score = np.clip(cyto_s - bg, 0, 1)

    fg_vals = fg_score[fg_score > 0]
    if fg_vals.size >= 500:
        thr_base = float(filters.threshold_otsu(fg_vals))
    else:
        thr_base = 0.15

    thr_weak = max(0.05, 0.70 * thr_base)

    weak_fg = fg_score > thr_weak
    weak_fg = morphology.binary_closing(weak_fg, morphology.disk(4))
    weak_fg = morphology.remove_small_objects(weak_fg, min_size=120)

    edge_map = sobel(cyto_s)
    edge_strength = float(edge_map[weak_fg].mean()) if np.any(weak_fg) else 0.0
    contrast = float(np.percentile(fg_score, 99) - np.percentile(fg_score, 50))

    lab = measure.label(weak_fg, connectivity=2)
    region_areas = np.array([r.area for r in measure.regionprops(lab)], dtype=np.float32)
    small_component_frac = float(np.mean(region_areas < 250)) if region_areas.size else 1.0

    H, W = cyto_img.shape[:2]
    ring_mean_scores = []
    ring_fill_scores = []
    hit_count = 0
    touch_areas = []

    for cx, cy in centers:
        rr_o, cc_o = draw_disk((cy, cx), annulus_outer_r, shape=(H, W))
        rr_i, cc_i = draw_disk((cy, cx), annulus_inner_r, shape=(H, W))

        ring = np.zeros((H, W), dtype=bool)
        ring[rr_o, cc_o] = True
        ring[rr_i, cc_i] = False

        vals = fg_score[ring]
        if vals.size == 0:
            continue

        ring_mean_scores.append(float(vals.mean()))
        ring_fill_scores.append(float((vals > thr_weak).mean()))

        touch_labels = lab[ring]
        touch_labels = touch_labels[touch_labels > 0]
        if touch_labels.size > 0:
            hit_count += 1
            uniq, counts = np.unique(touch_labels, return_counts=True)
            best_lab = uniq[np.argmax(counts)]
            touch_areas.append(int((lab == best_lab).sum()))

    ring_mean = float(np.mean(ring_mean_scores)) if ring_mean_scores else 0.0
    ring_fill = float(np.mean(ring_fill_scores)) if ring_fill_scores else 0.0
    hit_frac = float(hit_count / max(len(centers), 1))
    area_score = float(min(np.median(touch_areas) / 4000.0, 1.0)) if touch_areas else 0.0
    fg_frac = float(weak_fg.mean())

    fg_frac_penalty = 0.0
    if fg_frac < 0.01 or fg_frac > 0.40:
        fg_frac_penalty = 0.25

    score = (
        2.5 * ring_mean
        + 2.0 * ring_fill
        + 2.0 * hit_frac
        + 0.75 * contrast
        + 0.50 * edge_strength
        + 0.50 * area_score
        - 0.50 * small_component_frac
        - fg_frac_penalty
    )

    return {
        "score": float(score),
        "ring_mean": ring_mean,
        "ring_fill": ring_fill,
        "hit_frac": hit_frac,
        "contrast": contrast,
        "edge_strength": edge_strength,
        "area_score": area_score,
        "small_component_frac": small_component_frac,
        "fg_frac": fg_frac,
        "debug": {
            "cyto_norm": cyto_norm,
            "fg_score": fg_score,
            "weak_fg": weak_fg,
            "edge_map": edge_map,
        },
    }

def choose_best_cyto_channel(
    files_for_frame,
    nucleus_img,
    requested_channel="auto",
    preference=("647", "594", "488"),
):
    candidate_channels = sorted([ch for ch in files_for_frame.keys() if ch != "DAPI"])
    if len(candidate_channels) == 0:
        raise ValueError(f"No cytoplasm channels found. Available={sorted(files_for_frame.keys())}")

    bbox_nucleus = f.AppIntDINOwrapper(nucleus_img)
    centers = nucleus_boxes_to_centers(bbox_nucleus)

    rows = []
    for ch in candidate_channels:
        cyto_img = open_tiff_file(files_for_frame[ch])
        score_row = score_cyto_channel(cyto_img, centers)
        score_row["channel"] = ch
        score_row["path"] = str(files_for_frame[ch])
        rows.append(score_row)

    pref_rank = {ch: len(preference) - i for i, ch in enumerate(preference)}
    rows = sorted(rows, key=lambda r: (r["score"], pref_rank.get(r["channel"], 0)), reverse=True)

    if requested_channel is not None and str(requested_channel).lower() != "auto":
        best_row = next((r for r in rows if r["channel"] == requested_channel), None)
        if best_row is None:
            raise ValueError(
                f"Requested channel={requested_channel} not found. "
                f"Available cytoplasm channels={candidate_channels}"
            )
    else:
        best_row = rows[0]

    return {
        "best_channel": best_row["channel"],
        "bbox_nucleus": bbox_nucleus,
        "centers": centers,
        "rows": rows,
    }

def channel_score_table(selection_info):
    cols = [
        "channel", "score",
        "ring_mean", "ring_fill", "hit_frac",
        "contrast", "edge_strength",
        "area_score", "small_component_frac", "fg_frac"
    ]
    return (
        pd.DataFrame([{k: row[k] for k in cols} for row in selection_info["rows"]])
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )


### segmentation logic

In [6]:
def segment_channel(nucleus, channel, bbox_nucleus=None):
    cyt_removed = remove_outliers(channel)
    cyt_norm = normalize_to_uint8(cyt_removed)

    cyt_clahe = clahe_filter(cyt_norm)

    sigma_est = calculate_sigma(cyt_clahe, verbose=False)
    sigma_norm = max(float(sigma_est + 3.0), 1.0)
    sigma_weak = max(float(sigma_est - 10.0), 1.0)

    cyt_bilat = cv2.bilateralFilter(
        cyt_clahe,
        d=9,
        sigmaColor=sigma_norm,
        sigmaSpace=15,
        borderType=cv2.BORDER_REFLECT_101,
    )
    cyt_edge_preserved = edge_preserve_gray(cyt_clahe, sigma_s=sigma_norm, sigma_r=0.4)
    cyt_bilat_edge = edge_preserve_gray(cyt_bilat, sigma_s=sigma_weak, sigma_r=0.4)
    cyt_blended = laplacian(cyt_norm, cyt_bilat)

    cyt_rgb = np.stack([cyt_blended, cyt_bilat, cyt_edge_preserved], axis=-1)

    if bbox_nucleus is None:
        bbox_nucleus = f.AppIntDINOwrapper(nucleus)

    centers = nucleus_boxes_to_centers(bbox_nucleus)
    masks_bilat_edge = watershed_segment_with_centers(cyt_bilat_edge, centers)

    bbox_cyto = [mask_to_bbox(m) for m in masks_bilat_edge]
    bbox_cyto = [b for b in bbox_cyto if b is not None]
    bbox_cyto = [
        [float(x1), float(y1), float(x2), float(y2)]
        for (x1, y1, x2, y2) in bbox_cyto
    ]

    masks_all = f.finetune.AppIntPREDICTCytoplasmWrapper(cyt_rgb, bbox_cyto)

    masks = []
    for mset in masks_all:
        areas = [(mm > 0).sum() for mm in mset]
        best = mset[int(np.argmax(areas))]
        masks.append(postproc_mask(best))

    return masks, centers, cyt_clahe


### instance metric 

In [7]:
def instance_masks(lbl):
    ids = np.unique(lbl)
    ids = ids[ids > 0]
    out = []
    for k in ids:
        m = (lbl == k)
        if m.any():
            out.append((int(k), m))
    return out

def masks_to_label(masks, shape):
    lbl = np.zeros(shape, dtype=np.int32)
    for i, m in enumerate(masks, start=1):
        mb = np.asarray(m).astype(bool)
        lbl[mb] = i
    return lbl

def masked_iou(a, b, valid=None, eps=1e-8):
    if valid is not None:
        a = a & valid
        b = b & valid
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union + eps)

def masked_dice(a, b, valid=None, eps=1e-8):
    if valid is not None:
        a = a & valid
        b = b & valid
    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()
    return float(2.0 * inter) / float(denom + eps)

def calculate_instance_metrics_with_ignore_debug(
    gt_lbl,
    pred_lbl,
    ignore_mask,
    ignore_frac=0.60,
    iou_thresh=0.50,
):
    ignb = np.asarray(ignore_mask).astype(bool)
    valid = ~ignb

    gt_instances = instance_masks(gt_lbl)
    pred_instances_all = instance_masks(pred_lbl)

    pred_rows = []
    kept_preds = []
    dropped_pred_ids = []

    for pid, pm in pred_instances_all:
        area = int(pm.sum())
        if area == 0:
            continue

        ignore_pixels = int((pm & ignb).sum())
        frac = float(ignore_pixels) / float(area)

        dropped = frac >= ignore_frac
        pred_rows.append({
            "pred_id": int(pid),
            "area": area,
            "ignore_pixels": ignore_pixels,
            "ignore_overlap_frac": frac,
            "dropped": bool(dropped),
        })

        if dropped:
            dropped_pred_ids.append(int(pid))
        else:
            kept_preds.append((int(pid), pm))

    gt_used = set()
    pred_used = set()
    matches = []

    pairs = []
    for gi, (gid, gm) in enumerate(gt_instances):
        for pj, (pid, pm) in enumerate(kept_preds):
            iou = masked_iou(gm, pm, valid=valid)
            if iou >= iou_thresh:
                pairs.append((iou, gi, pj))

    pairs.sort(reverse=True, key=lambda x: x[0])

    for iou, gi, pj in pairs:
        gid = gt_instances[gi][0]
        pid = kept_preds[pj][0]

        if gid in gt_used or pid in pred_used:
            continue

        gt_used.add(gid)
        pred_used.add(pid)

        gm = gt_instances[gi][1]
        pm = kept_preds[pj][1]
        dice = masked_dice(gm, pm, valid=valid)

        matches.append({
            "gt_id": int(gid),
            "pred_id": int(pid),
            "iou": float(iou),
            "dice": float(dice),
        })

    TP = len(matches)
    FP = len(kept_preds) - TP
    FN = len(gt_instances) - TP

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    mean_iou = float(np.mean([m["iou"] for m in matches])) if matches else 0.0
    mean_dice = float(np.mean([m["dice"] for m in matches])) if matches else 0.0

    kept_pred_ids = [pid for pid, _ in kept_preds]
    matched_pred_ids = sorted([m["pred_id"] for m in matches])
    matched_gt_ids = sorted([m["gt_id"] for m in matches])
    unmatched_pred_ids = sorted([pid for pid in kept_pred_ids if pid not in matched_pred_ids])
    unmatched_gt_ids = sorted([gid for gid, _ in gt_instances if gid not in matched_gt_ids])

    return {
        "raw_pred": len(pred_instances_all),
        "kept": len(kept_preds),
        "dropped": len(dropped_pred_ids),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "mean_iou": mean_iou,
        "mean_dice": mean_dice,
        "matches": matches,
        "pred_rows": pred_rows,
        "kept_pred_ids": kept_pred_ids,
        "dropped_pred_ids": dropped_pred_ids,
        "matched_pred_ids": matched_pred_ids,
        "matched_gt_ids": matched_gt_ids,
        "unmatched_pred_ids": unmatched_pred_ids,
        "unmatched_gt_ids": unmatched_gt_ids,
    }


### ignore_mask build

In [9]:
def build_ignore_mask_from_signal_simple(
    nucleus_img,
    cyto_img,
    gt_lbl,
    smooth_sigma=1.0,
    bg_disk_radius=25,
    min_fg_pixels_for_otsu=500,
    connect_dist=8,
    min_component_area=300,
    max_component_area=200000,
    use_nucleus_gate=True,
    nuc_gate_dilate_radius=18,
    min_nuc_overlap_pixels=10,
    force_ignore_nuclei=True,
    nuc_force_r=6,
    ignore_min_area_final=800,
    prune_open_r=2,
    gt_near_radius=0,
    return_debug=False,
):
    gt = np.asarray(gt_lbl) > 0

    cyto = robust_norm(cyto_img)
    nuc = robust_norm(nucleus_img)

    cyto_s = gaussian(cyto, sigma=float(smooth_sigma), preserve_range=True)
    bg = morphology.opening(cyto_s, morphology.disk(int(bg_disk_radius)))
    fg_score = np.clip(cyto_s - bg, 0, 1)

    fg_vals = fg_score[fg_score > 0]
    if fg_vals.size >= int(min_fg_pixels_for_otsu):
        thr_fg = float(filters.threshold_otsu(fg_vals))
    else:
        thr_fg = 0.15

    fg_raw = fg_score > thr_fg
    closing_r = max(1, int(np.ceil(float(connect_dist) / 2.0)))
    fg_closed = morphology.binary_closing(fg_raw, morphology.disk(closing_r))
    unlabeled = fg_closed & (~gt)

    nuc_bin = None
    nuc_gate = None

    if use_nucleus_gate:
        nuc_vals = nuc[nuc > 0]
        if nuc_vals.size >= int(min_fg_pixels_for_otsu):
            thr_nuc = float(filters.threshold_otsu(nuc_vals))
        else:
            thr_nuc = 0.25

        nuc_bin = nuc > thr_nuc
        nuc_bin = morphology.remove_small_objects(nuc_bin, min_size=30)
        nuc_gate = morphology.binary_dilation(
            nuc_bin,
            morphology.disk(int(nuc_gate_dilate_radius)),
        )

    lab = measure.label(unlabeled, connectivity=2)
    ignore = np.zeros(gt.shape, dtype=bool)

    for region in measure.regionprops(lab):
        area = int(region.area)
        if area < int(min_component_area):
            continue
        if area > int(max_component_area):
            continue

        rr, cc = region.coords[:, 0], region.coords[:, 1]

        if use_nucleus_gate and (nuc_gate is not None):
            overlap = int(np.count_nonzero(nuc_gate[rr, cc]))
            if overlap < int(min_nuc_overlap_pixels):
                continue

        ignore[rr, cc] = True

    if force_ignore_nuclei and use_nucleus_gate and (nuc_bin is not None) and int(nuc_force_r) > 0:
        nuc_force = morphology.binary_dilation(nuc_bin, morphology.disk(int(nuc_force_r)))
        ignore |= nuc_force

    if int(ignore_min_area_final) > 0:
        ignore = morphology.remove_small_objects(ignore, min_size=int(ignore_min_area_final))

    if int(prune_open_r) > 0:
        ignore = morphology.binary_opening(ignore, morphology.disk(int(prune_open_r)))

    if int(gt_near_radius) > 0:
        gt_band = morphology.binary_dilation(gt, morphology.disk(int(gt_near_radius)))
        ignore[gt_band] = False

    ignore[gt] = False
    ignore = ignore.astype(np.uint8)

    meta = {
        "source": "built_from_signal",
        "ignore_frac": float(ignore.mean()),
        "fg_frac": float(fg_closed.mean()),
        "gt_frac": float(gt.mean()),
    }

    if return_debug:
        debug = {
            "cyto_s": cyto_s,
            "bg": bg,
            "fg_score": fg_score,
            "fg_raw": fg_raw,
            "fg_closed": fg_closed,
            "unlabeled": unlabeled,
            "nuc_gate": nuc_gate,
            "ignore": ignore,
        }
        return ignore, meta, debug

    return ignore, meta

def find_finalized_seg(sample_id, frame_id, finalized_dir=FINALIZED_DIR):
    frame_id = str(frame_id).zfill(3)
    for split in ("train", "test"):
        p = finalized_dir / split / sample_id / f"{sample_id}_img_{frame_id}_seg.npy"
        if p.exists():
            return p
    return None

def load_or_build_ignore_mask(
    sample_id,
    frame_id,
    nucleus_img,
    cyto_img,
    gt_lbl,
    finalized_dir=FINALIZED_DIR,
    use_finalized_ignore=True,
):
    if use_finalized_ignore:
        seg_path = find_finalized_seg(sample_id, frame_id, finalized_dir)
        if seg_path is not None:
            d = np.load(str(seg_path), allow_pickle=True).item()
            ignore_mask = (np.asarray(d.get("ignore_mask", np.zeros_like(gt_lbl))) > 0).astype(np.uint8)
            meta = {
                "source": "finalized_seg",
                "ignore_frac": float(ignore_mask.mean()),
                "seg_path": str(seg_path),
            }
            meta.update(d.get("ignore_meta", {}))
            return ignore_mask, meta, None

    ignore_mask, meta, debug = build_ignore_mask_from_signal_simple(
        nucleus_img=nucleus_img,
        cyto_img=cyto_img,
        gt_lbl=gt_lbl,
        return_debug=True,
    )
    return ignore_mask, meta, debug


### plotting

In [ ]:
def mask_centroid(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return float(xs.mean()), float(ys.mean())

def draw_binary_contours(ax, mask, color, linewidth=1.5):
    mask = np.asarray(mask).astype(bool)
    if not mask.any():
        return
    for contour in measure.find_contours(mask.astype(np.uint8), 0.5):
        ax.plot(contour[:, 1], contour[:, 0], color=color, linewidth=linewidth)

def draw_label_contours(ax, lbl, ids, color, linewidth=1.5):
    for k in ids:
        draw_binary_contours(ax, lbl == k, color=color, linewidth=linewidth)

def plot_channel_selection_debug(selection_info, title=""):
    rows = selection_info["rows"]
    n = len(rows)

    fig, ax = plt.subplots(2, n, figsize=(5 * n, 10))
    if n == 1:
        ax = np.asarray(ax).reshape(2, 1)

    for i, row in enumerate(rows):
        is_best = row["channel"] == selection_info["best_channel"]
        tag = "SELECTED" if is_best else ""

        ax[0, i].imshow(row["debug"]["cyto_norm"], cmap="gray", vmin=0, vmax=1)
        ax[0, i].set_title(f"{row['channel']} {tag}\nscore={row['score']:.3f}")
        ax[0, i].axis("off")

        ax[1, i].imshow(row["debug"]["fg_score"], cmap="magma")
        ax[1, i].set_title(
            f"ring_mean={row['ring_mean']:.3f}\n"
            f"ring_fill={row['ring_fill']:.3f}\n"
            f"hit_frac={row['hit_frac']:.3f}\n"
            f"small_frac={row['small_component_frac']:.3f}"
        )
        ax[1, i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_ignore_debug(nucleus_img, cyto_img, gt_lbl, ignore_mask, debug, title=""):
    if debug is None:
        print("No ignore-build debug available because ignore mask was loaded from finalized data.")
        return

    fig, ax = plt.subplots(2, 4, figsize=(22, 10))

    ax[0, 0].imshow(robust_norm(nucleus_img), cmap="gray")
    ax[0, 0].set_title("Nucleus")
    ax[0, 0].axis("off")

    ax[0, 1].imshow(robust_norm(cyto_img), cmap="gray")
    ax[0, 1].set_title("Selected cytoplasm")
    ax[0, 1].axis("off")

    ax[0, 2].imshow(debug["fg_score"], cmap="magma")
    ax[0, 2].set_title("FG score")
    ax[0, 2].axis("off")

    ax[0, 3].imshow(debug["fg_raw"], cmap="gray")
    ax[0, 3].set_title("FG raw")
    ax[0, 3].axis("off")

    ax[1, 0].imshow(debug["fg_closed"], cmap="gray")
    ax[1, 0].set_title("FG closed")
    ax[1, 0].axis("off")

    ax[1, 1].imshow(debug["unlabeled"], cmap="gray")
    ax[1, 1].set_title("Unlabeled FG")
    ax[1, 1].axis("off")

    nuc_gate = debug["nuc_gate"]
    ax[1, 2].imshow(np.zeros_like(ignore_mask) if nuc_gate is None else nuc_gate, cmap="gray")
    ax[1, 2].set_title("Nucleus gate")
    ax[1, 2].axis("off")

    ax[1, 3].imshow(robust_norm(cyto_img), cmap="gray")
    ax[1, 3].imshow(ignore_mask, cmap="Reds", alpha=0.35, vmin=0, vmax=1)
    draw_label_contours(ax[1, 3], gt_lbl, np.unique(gt_lbl)[np.unique(gt_lbl) > 0], color="cyan", linewidth=1.0)
    ax[1, 3].set_title("Final ignore + GT contours")
    ax[1, 3].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_frame_evaluation(bundle, title="", savepath=None, show=True):
    gt_lbl = bundle["gt_lbl"]
    pred_lbl = bundle["pred_lbl"]
    ignore_mask = bundle["ignore_mask"]
    cyto_vis = bundle["cyto_vis"]
    centers = bundle["centers"]
    metrics = bundle["metrics"]

    pred_rows = metrics["pred_rows"]
    dropped_ids = metrics["dropped_pred_ids"]
    matched_pred_ids = metrics["matched_pred_ids"]
    unmatched_pred_ids = metrics["unmatched_pred_ids"]

    dropped_frac_map = {
        row["pred_id"]: row["ignore_overlap_frac"]
        for row in pred_rows
        if row["dropped"]
    }

    gt_ids = [int(x) for x in np.unique(gt_lbl) if x > 0]
    pred_ids = [int(x) for x in np.unique(pred_lbl) if x > 0]

    cmap_lbl = get_instance_cmap()
    fig, ax = plt.subplots(1, 5, figsize=(32, 7))

    ax[0].imshow(cyto_vis, cmap="gray")
    draw_label_contours(ax[0], pred_lbl, pred_ids, color="orange", linewidth=1.2)
    for (cx, cy) in centers:
        ax[0].add_patch(Circle((cx, cy), radius=4, color="lime", fill=True))
    ax[0].set_title("Seg input + pred outlines")
    ax[0].axis("off")

    ax[1].imshow(gt_lbl, cmap=cmap_lbl, interpolation="nearest", vmin=0.5)
    for gid in gt_ids:
        c = mask_centroid(gt_lbl == gid)
        if c is not None:
            ax[1].annotate(
                str(gid),
                (c[0], c[1]),
                color="black",
                fontsize=11,
                fontweight="bold",
                ha="center",
                va="center",
                zorder=20,
                clip_on=False,
            )
    ax[1].set_title(f"GT labels (n={int(gt_lbl.max())})")
    ax[1].axis("off")

    ax[2].imshow(pred_lbl, cmap=cmap_lbl, interpolation="nearest", vmin=0.5)
    for pid in pred_ids:
        c = mask_centroid(pred_lbl == pid)
        if c is not None:
            ax[2].annotate(
                str(pid),
                (c[0], c[1]),
                color="black",
                fontsize=11,
                fontweight="bold",
                ha="center",
                va="center",
                zorder=20,
                clip_on=False,
            )
    ax[2].set_title(f"Pred labels (n={len(pred_rows)})")
    ax[2].axis("off")

    ax[3].imshow(cyto_vis, cmap="gray")
    ax[3].imshow(ignore_mask, cmap="Reds", alpha=0.35, vmin=0, vmax=1)
    ax[3].set_title(f"Ignore mask\n{float(ignore_mask.mean())*100:.2f}%")
    ax[3].axis("off")

    ax[4].imshow(cyto_vis, cmap="gray")
    draw_label_contours(ax[4], gt_lbl, gt_ids, color="cyan", linewidth=1.5)
    draw_label_contours(ax[4], pred_lbl, matched_pred_ids, color="yellow", linewidth=2.0)
    draw_label_contours(ax[4], pred_lbl, unmatched_pred_ids, color="magenta", linewidth=2.0)
    draw_label_contours(ax[4], pred_lbl, dropped_ids, color="red", linewidth=2.0)

    for pid in dropped_ids:
        frac = dropped_frac_map.get(pid, None)
        c = mask_centroid(pred_lbl == pid)
        if frac is not None and c is not None:
            ax[4].annotate(
                f"{pid}\n{frac:.2f}",
                (c[0], c[1]),
                color="black",
                fontsize=9,
                fontweight="bold",
                ha="center",
                va="center",
                zorder=20,
                clip_on=False,
            )

    ax[4].set_title("GT cyan | matched yellow | unmatched magenta | dropped red")
    ax[4].axis("off")

    plt.suptitle(title)
    plt.subplots_adjust(wspace=0.08)

    if savepath is not None:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, dpi=150, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)



### csv export

In [13]:
def _ids_to_str(values):
    values = list(values)
    return ",".join(str(v) for v in values) if values else ""

def frame_result_to_row(frame_result):
    metrics = frame_result["metrics"]
    ignore_meta = frame_result["ignore_meta"]

    return {
        "track_key": frame_result["track_key"],
        "sample_id": frame_result["sample_id"],
        "frame_index": int(frame_result["frame_index"]),
        "frame_id": str(frame_result["frame_id"]),
        "fname": frame_result["fname"],
        "available_channels": ",".join(frame_result["available_channels"]),
        "selected_cyto_channel": frame_result["selected_cyto_channel"],
        "ignore_source": ignore_meta.get("source", ""),
        "ignore_source_channel": ignore_meta.get("ignore_source_channel", ""),
        "ignore_source_score": ignore_meta.get("ignore_source_score", np.nan),
        "ignore_frac": float(ignore_meta.get("ignore_frac", np.asarray(frame_result["ignore_mask"]).astype(bool).mean())),
        "n_gt_instances": int(frame_result["gt_lbl"].max()),
        "raw_pred": int(metrics["raw_pred"]),
        "kept": int(metrics["kept"]),
        "dropped": int(metrics["dropped"]),
        "TP": int(metrics["TP"]),
        "FP": int(metrics["FP"]),
        "FN": int(metrics["FN"]),
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "mean_iou": float(metrics["mean_iou"]),
        "mean_dice": float(metrics["mean_dice"]),
        "matched_pred_ids": _ids_to_str(metrics["matched_pred_ids"]),
        "unmatched_pred_ids": _ids_to_str(metrics["unmatched_pred_ids"]),
        "dropped_pred_ids": _ids_to_str(metrics["dropped_pred_ids"]),
        "matched_gt_ids": _ids_to_str(metrics["matched_gt_ids"]),
        "unmatched_gt_ids": _ids_to_str(metrics["unmatched_gt_ids"]),
    }


## Finalized Inspection

In [14]:
seg_path = FINALIZED_DIR / "train" / "EXP251009COLA01" / "EXP251009COLA01_img_001_seg.npy"
d = np.load(str(seg_path), allow_pickle=True).item()

rows = []
for k, v in d.items():
    row = {"key": k, "python_type": type(v).__name__}

    if isinstance(v, np.ndarray):
        row["dtype"] = str(v.dtype)
        row["shape"] = tuple(v.shape)
        row["summary"] = f"min={np.min(v)} max={np.max(v)}"
    elif isinstance(v, dict):
        row["dtype"] = "-"
        row["shape"] = "-"
        row["summary"] = f"{len(v)} keys"
    elif isinstance(v, (list, tuple)):
        row["dtype"] = "-"
        row["shape"] = f"len={len(v)}"
        row["summary"] = str(v[:5]) if len(v) > 0 else "[]"
    else:
        row["dtype"] = "-"
        row["shape"] = "-"
        row["summary"] = str(v)

    rows.append(row)

display(pd.DataFrame(rows).sort_values("key").reset_index(drop=True))


,key,python_type,dtype,shape,summary
0,available_channels,list,-,len=3,"['488', '647', 'DAPI']"
1,channel_files,dict,-,-,3 keys
2,channels,list,-,len=2,"[1, 0]"
3,flows,ndarray,float32,"(2, 2048, 2048)",min=0.0 max=0.0
4,frame_id,str,-,-,001
5,ignore_mask,ndarray,uint8,"(2048, 2048)",min=0 max=1
6,ignore_meta,dict,-,-,17 keys
7,ismanual,ndarray,uint8,"(13,)",min=1 max=1
8,masks,ndarray,int32,"(2048, 2048)",min=0 max=12
9,outlines,ndarray,int32,"(2048, 2048)",min=0 max=0


In [15]:
print("ignore_meta")
display(pd.DataFrame([d.get("ignore_meta", {})]).T.rename(columns={0: "value"}))

print("\nchannel_files")
display(pd.DataFrame([d.get("channel_files", {})]).T.rename(columns={0: "path"}))


ignore_meta


,value
source_method,build_ignore_mask_from_signal_new
ignore_source_channel,647
ignore_source_score,83.0
channel_order_for_ignore,"[DAPI, 647]"
thr_fg,0.192215
thr_fg_method,otsu
thr_nuc,0.395742
thr_nuc_method,otsu
connect_dist,10
closing_radius,5



channel_files


,path
DAPI,Z:\Guohao\For Computation group\EXP251009COL_f...
647,Z:\Guohao\For Computation group\EXP251009COL_f...
488,Z:\Guohao\For Computation group\EXP251009COL_f...


In [16]:
def run_one_frame(
    track_key,
    frame_index=None,
    frame_id=None,
    cyto_channel="auto",
    use_finalized_ignore=True,
):
    sample_id = track_key_to_sample_id(track_key)
    sample_dir = BASE_DIR / sample_id
    frame_file_map = build_frame_file_map(sample_dir)

    track = mat[track_key]
    num_frames = track.shape[1]

    if frame_id is not None:
        frame_id = str(frame_id).zfill(3)

    if frame_index is None:
        if frame_id is None:
            raise ValueError("Provide either frame_index or frame_id.")
        found = None
        for idx in range(num_frames):
            frame_obj = track[0, idx][0, 0]
            sid = parse_frame_id(extract_filename(frame_obj.filename))
            if sid == frame_id:
                found = idx
                break
        if found is None:
            raise ValueError(f"Could not find frame_id={frame_id} in {track_key}")
        frame_index = found

    frame_obj = track[0, frame_index][0, 0]
    fname = extract_filename(frame_obj.filename)
    sid = parse_frame_id(fname)

    files_for_frame = frame_file_map.get(sid, {})
    if "DAPI" not in files_for_frame:
        raise FileNotFoundError(
            f"Missing DAPI for {sample_id} s{sid}. Available={sorted(files_for_frame.keys())}"
        )

    nucleus = open_tiff_file(files_for_frame["DAPI"])

    selection_info = choose_best_cyto_channel(
        files_for_frame=files_for_frame,
        nucleus_img=nucleus,
        requested_channel=cyto_channel,
    )
    selected_cyto_channel = selection_info["best_channel"]
    cyto = open_tiff_file(files_for_frame[selected_cyto_channel])

    masks_pred, centers, cyto_vis = segment_channel(
        nucleus,
        cyto,
        bbox_nucleus=selection_info["bbox_nucleus"],
    )

    cells = extract_cells_from_frame(frame_obj)
    gt_lbl = reconstruct_instance_mask(cells, H=nucleus.shape[0], W=nucleus.shape[1])
    pred_lbl = masks_to_label(masks_pred, gt_lbl.shape)

    ignore_mask, ignore_meta, ignore_debug = load_or_build_ignore_mask(
        sample_id=sample_id,
        frame_id=sid,
        nucleus_img=nucleus,
        cyto_img=cyto,
        gt_lbl=gt_lbl,
        finalized_dir=FINALIZED_DIR,
        use_finalized_ignore=use_finalized_ignore,
    )

    metrics = calculate_instance_metrics_with_ignore_debug(
        gt_lbl,
        pred_lbl,
        ignore_mask,
        ignore_frac=IGNORE_OVERLAP_THRESH,
        iou_thresh=MATCH_IOU_THRESH,
    )

    return {
        "track_key": track_key,
        "sample_id": sample_id,
        "frame_index": frame_index,
        "frame_id": sid,
        "fname": fname,
        "available_channels": sorted(files_for_frame.keys()),
        "selected_cyto_channel": selected_cyto_channel,
        "channel_files": {k: str(v) for k, v in files_for_frame.items()},
        "channel_selection": selection_info,
        "nucleus": nucleus,
        "cyto": cyto,
        "cyto_vis": cyto_vis,
        "centers": centers,
        "masks_pred": masks_pred,
        "pred_lbl": pred_lbl,
        "gt_lbl": gt_lbl,
        "ignore_mask": ignore_mask,
        "ignore_meta": ignore_meta,
        "ignore_debug": ignore_debug,
        "metrics": metrics,
    }


In [17]:
pd.set_option("display.max_columns", None)

TRACK_KEY = "Track251009COLA01"
FRAME_ID = "001"
FRAME_INDEX = None

frame_result = run_one_frame(
    TRACK_KEY,
    frame_index=FRAME_INDEX,
    frame_id=FRAME_ID,
    cyto_channel="auto",
    use_finalized_ignore=True,
)

print("Available channels:", frame_result["available_channels"])
print("Selected cytoplasm channel:", frame_result["selected_cyto_channel"])

display(channel_score_table(frame_result["channel_selection"]))

plot_channel_selection_debug(
    frame_result["channel_selection"],
    title=f"{frame_result['sample_id']} s{frame_result['frame_id']} | Channel selection",
)

plot_frame_evaluation(
    frame_result,
    title=f"{frame_result['sample_id']} s{frame_result['frame_id']} | instance evaluation",
)

frame_summary_df = pd.DataFrame([frame_result_to_row(frame_result)])
display(frame_summary_df)

# Optional detailed per-prediction table
pred_detail_df = (
    pd.DataFrame(frame_result["metrics"]["pred_rows"])
    .sort_values("ignore_overlap_frac", ascending=False)
    .reset_index(drop=True)
)
display(pred_detail_df)


DEBUG: Running dino_bbox


DEBUG: DINO predicted 54 boxes
[CYTO] Starting prediction for 18 cells
[CYTO] Input image shape: (2048, 2048, 3), dtype: uint8
[CYTO] Using pre-formatted 3-channel image as RGB
[CYTO] Processing box [1141.0, 1693.0, 1427.0, 1958.0]
[CYTO] Processing box [1487.0, 1355.0, 1698.0, 1553.0]
[CYTO] Processing box [1052.0, 1029.0, 1255.0, 1241.0]


KeyboardInterrupt: 

## Batch Run

In [ ]:
OUTPUT_DIR = BASE_DIR / "metric_computation" / "watershed+SAM"
PANELS_DIR = OUTPUT_DIR / "panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PANELS_DIR.mkdir(parents=True, exist_ok=True)

all_rows = []
error_rows = []

for track_key in track_keys:
    track = mat[track_key]
    num_frames = track.shape[1]
    print(f"\n=== {track_key} | {num_frames} frames ===")

    for idx in range(num_frames):
        try:
            bundle = run_one_frame(
                track_key,
                frame_index=idx,
                cyto_channel="auto",
                use_finalized_ignore=True,
            )

            row = frame_result_to_row(bundle)
            all_rows.append(row)

            panel_path = (
                PANELS_DIR
                / bundle["sample_id"]
                / f"{bundle['sample_id']}_img_{bundle['frame_id']}_panel.png"
            )

            plot_frame_evaluation(
                bundle,
                title=(
                    f"{bundle['sample_id']} s{bundle['frame_id']} | "
                    f"cyto={bundle['selected_cyto_channel']} | "
                    f"IoU={row['mean_iou']:.3f} | Dice={row['mean_dice']:.3f}"
                ),
                savepath=panel_path,
                show=False,
            )

            print(
                f"{bundle['sample_id']} s{bundle['frame_id']} | "
                f"saved panel | cyto={bundle['selected_cyto_channel']} | "
                f"IoU={row['mean_iou']:.3f} | Dice={row['mean_dice']:.3f}"
            )

        except Exception as e:
            error_rows.append({
                "track_key": track_key,
                "frame_index": idx,
                "error": repr(e),
            })
            print(f"ERROR {track_key} frame_index={idx}: {e}")

all_results_df = (
    pd.DataFrame(all_rows)
    .sort_values(["sample_id", "frame_id"])
    .reset_index(drop=True)
)

errors_df = pd.DataFrame(error_rows)

all_results_csv = OUTPUT_DIR / "all_frame_metrics.csv"
errors_csv = OUTPUT_DIR / "all_frame_metric_errors.csv"
all_results_df.to_csv(all_results_csv, index=False)
errors_df.to_csv(errors_csv, index=False)

overall_summary_df = pd.DataFrame([{
    "n_frames": int(len(all_results_df)),
    "n_errors": int(len(errors_df)),
    "mean_iou_all_frames": float(all_results_df["mean_iou"].mean()) if len(all_results_df) else np.nan,
    "mean_dice_all_frames": float(all_results_df["mean_dice"].mean()) if len(all_results_df) else np.nan,
    "mean_precision_all_frames": float(all_results_df["precision"].mean()) if len(all_results_df) else np.nan,
    "mean_recall_all_frames": float(all_results_df["recall"].mean()) if len(all_results_df) else np.nan,
    "mean_f1_all_frames": float(all_results_df["f1"].mean()) if len(all_results_df) else np.nan,
}])

per_sample_summary_df = (
    all_results_df
    .groupby("sample_id", as_index=False)[["mean_iou", "mean_dice", "precision", "recall", "f1"]]
    .mean()
    .sort_values("sample_id")
    .reset_index(drop=True)
)

overall_csv = OUTPUT_DIR / "overall_means.csv"
per_sample_csv = OUTPUT_DIR / "per_sample_means.csv"

overall_summary_df.to_csv(overall_csv, index=False)
per_sample_summary_df.to_csv(per_sample_csv, index=False)

print("\nSaved:")
print("Panels root:", PANELS_DIR)
print("Frame metrics:", all_results_csv)
print("Errors:", errors_csv)
print("Overall means:", overall_csv)
print("Per-sample means:", per_sample_csv)

print("\nOverall means across all frames in all directories")
display(overall_summary_df)



=== Track251009COLA01 | 5 frames ===
DEBUG: Running dino_bbox


DEBUG: DINO predicted 54 boxes
